In [2]:
import tqdm
import torch
import numpy as np
from bgflow.utils import as_numpy
from bgflow import DiffEqFlow, BoltzmannGenerator, MeanFreeNormalDistribution
from bgflow import BlackBoxDynamics, BruteForceEstimator
from tbg.models2 import EGNN_dynamics_AD2_cat


n_particles = 22
n_dimensions = 3
dim = n_particles * n_dimensions

scaling = 10

# atom types for backbone
n_particles = 22
n_dimensions = 3
dim = n_particles * n_dimensions


atom_types = np.arange(22)
atom_types[[1, 2, 3]] = 2
atom_types[[19, 20, 21]] = 20
atom_types[[11, 12, 13]] = 12
h_initial = torch.nn.functional.one_hot(torch.tensor(atom_types))


# now set up a prior
prior = MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False).cuda()
prior_cpu = MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False)

brute_force_estimator = BruteForceEstimator()
net_dynamics = EGNN_dynamics_AD2_cat(
    n_particles=n_particles,
    device="cuda",
    n_dimension=dim // n_particles,
    h_initial=h_initial,
    hidden_nf=64,
    act_fn=torch.nn.SiLU(),
    n_layers=5,
    recurrent=True,
    tanh=True,
    attention=True,
    condition_time=True,
    mode="egnn_dynamics",
    agg="sum",
)

bb_dynamics = BlackBoxDynamics(
    dynamics_function=net_dynamics, divergence_estimator=brute_force_estimator
)

flow = DiffEqFlow(dynamics=bb_dynamics)

bg = BoltzmannGenerator(prior, flow, prior).cuda()


class BruteForceEstimatorFast(torch.nn.Module):
    """
    Exact bruteforce estimation of the divergence of a dynamics function.
    """

    def __init__(self):
        super().__init__()
        self.n = 0

    def forward(self, dynamics, t, xs):
        self.n += 1

        with torch.set_grad_enabled(True):
            xs.requires_grad_(True)
            x = [xs[:, [i]] for i in range(xs.size(1))]

            dxs = dynamics(t, torch.cat(x, dim=1))

            assert len(dxs.shape) == 2, "`dxs` must have shape [n_btach, system_dim]"
            divergence = 0
            for i in range(xs.size(1)):
                divergence += torch.autograd.grad(
                    dxs[:, [i]], x[i], torch.ones_like(dxs[:, [i]]), retain_graph=True
                )[0]

        return dxs, -divergence.view(-1, 1)


brute_force_estimator_fast = BruteForceEstimatorFast()
# use OTD in the evaluation process
bb_dynamics._divergence_estimator = brute_force_estimator_fast
bg.flow._integrator_atol = 1e-4
bg.flow._integrator_rtol = 1e-4
flow._use_checkpoints = False
flow._kwargs = {}


filename = "Flow-Matching-AD2-amber-weighted-encoding"


PATH_last = f"models/{filename}"
checkpoint = torch.load(PATH_last)
flow.load_state_dict(checkpoint["model_state_dict"])


/tmp/ipykernel_1658791/2822256220.py:100: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH_last)


<All keys matched successfully>

In [3]:
# alpha_t = t
# dot alpha_t = 1
# beta_t = 1 - t
# dot beta_t = -1
# kappa_t = 1 / t
# eta_t = (1 - t) ( 1 / t * (1-t) - (-1))
# eta_t = (1 - t) ((1 - t) / t + t / t)
# eta_t = (1 - t) (1 / t)
# eta_t = (1 - t) / t

# f = 1 / t
# score = 1 / eta (vt - xt / t)

In [4]:
def stochastic_flow(t, xt):
    vt = flow_fn(t, xt)
    return 2 * vt - xt / torch.clip(t, min=1e-2)

In [5]:
flow_fn = flow._dynamics._dynamics._dynamics_function

In [6]:
%%time
import pdb

filename = "Flow-Matching-AD2-amber-weighted-encoding-SDE-5"
n_samples = 400
n_sample_batches = 500
latent_np = np.empty(shape=(0))
samples_np = np.empty(shape=(0))
dlogp_np = np.empty(shape=(0))
print(f"Start sampling with {filename}")
n_samples=10
for i in tqdm.tqdm(range(n_sample_batches)):
    
    x0 = prior.sample(400)
    xt = x0
    #logp = prior.energy(x0)
    logp = torch.zeros(x0.shape[0], device="cuda")
    eps = 1e-3
    with torch.no_grad():
        n_steps = 1000
        dt = 1 / n_steps
        for t in torch.linspace(0, 1, n_steps + 1):
            
            sigma_t_squared = (2 * (1 - t) / torch.clip(t, min=eps))
            # sigma_t = (2 (1 - t) / t) ** 0.5
            sigma_t = sigma_t_squared ** 0.5
            vt = flow._inverse_dynamics._dynamics._dynamics._dynamics_function(t, xt)
            # st is correct we checked
            st = 2 * vt - xt / torch.clip(t, min=1e-2)
            noise_t = sigma_t * torch.randn_like(xt) * (dt ** 0.5)
            dxt = st * dt + noise_t
            xt = xt + dxt
            score_t = (t * vt - xt) / torch.clip(1 - t, min=eps)
            dlogp = ((-xt.shape[-1] / torch.clip(t, min=eps) 
                      + (score_t * (-sigma_t_squared / 2 * score_t + dxt - xt / torch.clip(t, min=eps))).sum(-1)) * dt)
            dlogp = ((-xt.shape[-1] / torch.clip(t, min=eps) 
              + (sigma_t_squared / 2 * score_t * score_t).sum(-1)) * dt
             + (score_t * noise_t).sum(-1))
            logp = logp + dlogp
        
        #print("n / i = ", flow_fn.counter / (i+1))
        latent_np = np.append(latent_np, x0.detach().cpu().numpy())
        samples_np = np.append(samples_np, xt.detach().cpu().numpy())
        
        dlogp_np = np.append(dlogp_np, as_numpy(logp))

    latent_np = latent_np.reshape(-1, dim)
    samples_np = samples_np.reshape(-1, dim)
    np.savez(
        f"result_data/{filename}",
        latent_np=latent_np,
        samples_np=samples_np,
        dlogp_np=dlogp_np,
    )


Start sampling with Flow-Matching-AD2-amber-weighted-encoding-SDE-5


/home/mila/a/alexander.tong/active_inference/tbg/tbg/models2.py:395: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t = torch.tensor(t).to(xs)
100%|█████████████████████████████████████████████████████████████████| 500/500 [2:27:51<00:00, 17.74s/it]

CPU times: user 2h 26min 37s, sys: 56.2 s, total: 2h 27min 34s
Wall time: 2h 27min 51s


In [3]:
x0_base = prior.sample(400)

In [3]:
x0_base = prior.sample(400)

In [4]:
%%time
import pdb

filename = "Flow-Matching-AD2-amber-weighted-encoding-ODE-3"
n_samples = 400
n_sample_batches = 50
latent_np = np.empty(shape=(0))
samples_np = np.empty(shape=(0))
dlogp_np = np.empty(shape=(0))
print(f"Start sampling with {filename}")
n_samples=10
for i in tqdm.tqdm(range(n_sample_batches)):
    
    x0 = x0_base
    xt = x0
    #logp = prior.energy(x0)
    logp = torch.zeros(x0.shape[0], device="cuda")
    eps = 1e-3
    with torch.no_grad():
        n_steps = 1
        dt = 1 / n_steps

        
        #print(y.shape)
        #break
        for t in torch.linspace(0, 1, n_steps + 1):
            
            sigma_t_squared = (2 * (1 - t) / torch.clip(t, min=eps))
            # sigma_t = (2 (1 - t) / t) ** 0.5
            sigma_t = sigma_t_squared ** 0.5
            vt = flow_fn(t, xt)
            # st is correct we checked
            st = 2 * vt - xt / torch.clip(t, min=1e-2)
            noise_t = sigma_t * torch.randn_like(xt) * (dt ** 0.5)
            dxt = vt * dt + noise_t * 0
            xt = xt + dxt
            score_t = (t * vt - xt) / torch.clip(1 - t, min=eps)
            dlogp = ((-xt.shape[-1] / torch.clip(t, min=eps) 
                      + (score_t * (-sigma_t_squared / 2 * score_t + dxt - xt / torch.clip(t, min=eps))).sum(-1)) * dt)
            logp = 0 # logp + dlogp
        from torchdiffeq import odeint_adjoint
        ts = torch.linspace(0, 1, 2).cuda()
        xt = odeint_adjoint(flow._dynamics._dynamics._dynamics_function, x0, ts, method="dopri5", rtol=1e-4, atol=1e-4)[-1]
        #print("n / i = ", flow_fn.counter / (i+1))
        latent_np = np.append(latent_np, x0.detach().cpu().numpy())
        samples_np = np.append(samples_np, xt.detach().cpu().numpy())
        dlogp_np = np.append(dlogp_np, as_numpy(logp))

    latent_np = latent_np.reshape(-1, dim)
    samples_np = samples_np.reshape(-1, dim)
    np.savez(
        f"result_data/{filename}",
        latent_np=latent_np,
        samples_np=samples_np,
        dlogp_np=dlogp_np,
    )
    break


Start sampling with Flow-Matching-AD2-amber-weighted-encoding-ODE-3


/home/mila/a/alexander.tong/active_inference/tbg/tbg/models2.py:395: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t = torch.tensor(t).to(xs)
/home/mila/a/alexander.tong/active_inference/tbg/tbg/models2.py:395: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t = torch.tensor(t).to(xs)
  0%|                                                                      | 0/50 [00:01<?, ?it/s]

CPU times: user 1.29 s, sys: 150 ms, total: 1.44 s
Wall time: 1.38 s


In [129]:
x0.mean()

tensor(-2.8899e-10, device='cuda:0')

In [7]:
sde_traj[-1].std()

tensor(1.5640, device='cuda:0')

In [35]:
dlogp_np

array([-33.22525406, -27.49834442, -28.57772064, -38.61469269,
       -26.72375298, -29.27663803, -32.7167511 , -30.01943207,
       -31.57315254, -34.20825195, -23.26143646, -33.45377731,
       -29.17208862, -23.18746567, -39.13430023, -29.48268127,
       -30.62382317, -31.03500557, -37.07038116, -25.86792755,
       -29.53132629, -30.15475082, -27.43104744, -19.63040733,
       -41.61455536, -27.06830978, -33.40086746,  -6.91728115,
       -31.11883926, -29.21347046])

In [38]:
as_numpy(prior.energy(torch.from_numpy(latent_np).cuda()))

array([[29.67830832],
       [32.53566091],
       [24.50982983],
       [38.74380258],
       [30.03843713],
       [27.98790041],
       [40.83639711],
       [27.34880375],
       [39.95910326],
       [30.93008203],
       [32.01614975],
       [28.15633461],
       [24.61263243],
       [22.20644789],
       [33.70998054],
       [29.70856764],
       [29.97583735],
       [32.87514124],
       [48.91089883],
       [42.96505349],
       [27.17271042],
       [25.18732634],
       [19.11418944],
       [29.24187397],
       [53.4582729 ],
       [28.38797646],
       [24.07998392],
       [17.74714482],
       [33.80144489],
       [22.86275165]])

In [40]:
dlogp_np.reshape(-1,1)

array([[-33.22525406],
       [-27.49834442],
       [-28.57772064],
       [-38.61469269],
       [-26.72375298],
       [-29.27663803],
       [-32.7167511 ],
       [-30.01943207],
       [-31.57315254],
       [-34.20825195],
       [-23.26143646],
       [-33.45377731],
       [-29.17208862],
       [-23.18746567],
       [-39.13430023],
       [-29.48268127],
       [-30.62382317],
       [-31.03500557],
       [-37.07038116],
       [-25.86792755],
       [-29.53132629],
       [-30.15475082],
       [-27.43104744],
       [-19.63040733],
       [-41.61455536],
       [-27.06830978],
       [-33.40086746],
       [ -6.91728115],
       [-31.11883926],
       [-29.21347046]])

In [36]:
latent_np

array([[-0.05310838, -0.74233639,  0.09671155, ..., -0.64717489,
        -0.65883487,  0.02002692],
       [ 1.3443377 ,  0.86011159, -0.78500056, ...,  0.29150739,
        -0.71700203,  1.04623652],
       [-0.75242269,  0.85173291, -1.13871932, ...,  1.29495609,
         1.08495033,  0.60654551],
       ...,
       [ 0.63808346, -1.33710253, -0.1480996 , ...,  1.02378666,
        -0.75945032, -0.42805123],
       [ 0.55500823,  0.33216324, -0.36078322, ..., -0.16847499,
        -0.19262022,  0.05808135],
       [ 1.93812954, -0.38517433, -0.2861217 , ..., -0.44084623,
         0.44077414, -1.19645965]], shape=(30, 66))

In [6]:
filename = "Flow-Matching-AD2-amber-weighted-encoding-Base"
n_samples=400
latent_np = np.empty(shape=(0))
samples_np = np.empty(shape=(0))
dlogp_np = np.empty(shape=(0))
for i in tqdm.tqdm(range(n_sample_batches)):
    with torch.no_grad():
        samples, latent, dlogp = bg.sample(n_samples, with_latent=True, with_dlogp=True)
        print("n / i = ", flow_fn.counter / (i+1))
        latent_np = np.append(latent_np, latent.detach().cpu().numpy())
        samples_np = np.append(samples_np, samples.detach().cpu().numpy())

        dlogp_np = np.append(dlogp_np, as_numpy(dlogp))

    latent_np = latent_np.reshape(-1, dim)
    samples_np = samples_np.reshape(-1, dim)
    print(samples_np.std())
    np.savez(
        f"result_data/{filename}",
        latent_np=latent_np,
        samples_np=samples_np,
        dlogp_np=dlogp_np,
    )
    break

  0%|                                                                      | 0/50 [00:32<?, ?it/s]

n / i =  128.0
1.5710560942281198


In [11]:
samples_np.std()

np.float64(1.5710516092086217)

In [ ]:
}